# Texas RRC Primary Data Extraction — Production Disposition & Well/Lease/API Identification

**Pipeline stage:** Step 1 of the Texas RRC oil production data pipeline.

This notebook is the entry point of the pipeline. It downloads nothing itself — it *reads* a raw
data dump that you must download manually first (instructions below) — and produces two cleaned,
lease/well-level tables that every later notebook in the pipeline depends on:

1. **Production disposition data** — how each oil lease's produced oil and casinghead gas was
   disposed of each month (sold via pipeline/truck/tank car, flared, used for gas lift, etc.).
2. **Well ↔ lease ↔ API identification** — the mapping from individual wells to the lease they
   belong to, plus each well's 8-digit RRC API number, which is required to look up well
   coordinates later in the pipeline.

Both parts of this notebook share the same zip-reading and chunked-CSV-reading machinery, defined
once in Section 1 and reused throughout.

> **Note on scope:** This notebook intentionally processes **only**
> `OG_LEASE_CYCLE_DISP_DATA_TABLE.dsv` (the disposition table) from the raw dump. The companion
> table `OG_LEASE_CYCLE_DATA_TABLE.dsv` (raw monthly production volumes) is not used here — this
> pipeline is scoped to disposition data only.

## 0. Data Source & How to Download the Raw Data

All data used in this notebook comes from the **Texas Railroad Commission (RRC)**, the state agency
that regulates oil and gas production in Texas.

**Download page:** https://www.rrc.texas.gov/resource-center/research/data-sets-available-for-download/

### Step-by-step download instructions

1. Go to the RRC data download page linked above.
2. Find the dataset named **"Production Data Query (PDQ)"** (sometimes listed as "Production Data
   Query Dump" or similar — RRC occasionally renames sections of the page). This is a bulk export of
   the RRC's production and historical ledger databases.
3. Download the file. It arrives as a single zip archive — in this pipeline it is referred to as the
   **outer zip** and expected to be named `texas_pdq.zip`.
4. **Do not unzip it manually.** The archive has a nested structure that this notebook unzips
   in-memory:

   ```
   texas_pdq.zip                     ← outer zip (what you downloaded)
   └── PDQ_DSV.zip                   ← inner zip, lives inside the outer zip
       ├── OG_LEASE_CYCLE_DATA_TABLE.dsv        (not used in this notebook)
       ├── OG_LEASE_CYCLE_DISP_DATA_TABLE.dsv   ← used in Part 1 below
       ├── OG_WELL_COMPLETION_DATA_TABLE.dsv    ← used in Part 2 below
       └── ... (many other .dsv tables we don't use)
   ```

   Each `.dsv` file is a flat text file delimited by the `}` character (not a comma), encoded as
   `latin-1`. These are **not** standard CSVs — the custom reader functions below handle this.

5. Place the downloaded `texas_pdq.zip` on disk and point the `OUTER_ZIP` variable in the config
   cells below at it. In this codebase the convention is:

   ```
   <project_root>/data/raw/texas/texas_pdq.zip
   ```

6. **Size expectations:** the outer zip is roughly **~5 GB compressed**, and the tables inside expand
   to **>25 GB uncompressed**. Do not try to unzip the whole thing to disk — the chunked, in-memory
   readers in this notebook are specifically designed to avoid ever materializing the full uncompressed
   file. Processing can still take a while and use several GB of RAM; chunk sizes are tuned down
   below for machines with ~16 GB RAM.
7. **Update cadence:** RRC refreshes this dump **monthly**. Re-download periodically if you want the
   latest production months. Historical coverage goes back to **1993**.

### What this notebook does with it

This notebook only touches two of the many tables inside `PDQ_DSV.zip`:

| Table | Used in | Contents |
|---|---|---|
| `OG_LEASE_CYCLE_DISP_DATA_TABLE.dsv` | Part 1 | Monthly disposition (sale/use/loss) of oil and casinghead gas, per lease |
| `OG_WELL_COMPLETION_DATA_TABLE.dsv` | Part 2 | One row per well: which lease it belongs to, and its API number components |

Both are filtered down to **oil leases/wells only** (`OIL_GAS_CODE == "O"`). Gas captured in this
pipeline is *casinghead gas* — natural gas dissolved in crude oil and produced alongside it (often
flared) — not gas from dedicated gas wells.

## 1. Shared Setup (Imports & Logging)

Run this cell first, and re-run it after any kernel crash or restart — everything both Part 1 and
Part 2 need from the standard library / third-party packages is imported here once, since both parts
of this notebook rely on the same zip-reading and chunked-CSV-reading machinery.

- `zipfile` / `io.BytesIO` / `io.TextIOWrapper` — for reading the nested zip archive without
  extracting it to disk.
- `pandas` — for all tabular processing.
- `logging` — gives timestamped progress messages as each multi-million-row chunk is processed,
  which is important here since these files are large and slow to read.
- `gc` — explicit garbage collection calls are sprinkled throughout to keep peak memory down when
  processing large chunks.

In [1]:
# ── Imports + logging config ──────────────────────────────────────────────────
# Run this first after any kernel crash. Shared by both Part 1 and Part 2 below.

import zipfile
import logging
import warnings
import sys
import gc
from io import TextIOWrapper, BytesIO
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger(__name__)

log.info("✓ Shared imports + logging ready.")


09:37:37 [INFO] ✓ Shared imports + logging ready.


## 2. Part 1 — Production Disposition Data (`OG_LEASE_CYCLE_DISP_DATA_TABLE`)

**Goal:** extract, clean, and save the monthly oil/casinghead-gas disposition breakdown for every
oil lease in Texas.

The disposition table answers the question *"once oil/gas was produced from this lease this month,
where did it go?"* — e.g. how many barrels were sold via pipeline vs. truck, how much casinghead gas
was flared vs. sent to a processing plant, etc. Every disposition code in the raw table is a separate
column of volume for that disposal method; we rename these into human-readable column names and add
a couple of derived "roll-up" totals (e.g. total oil sold, total gas flared).

**Output of Part 1:** `texas_prod_disp.parquet` (or `.csv`, depending on `FORMAT` below).

Only the raw `OG_LEASE_CYCLE_DATA_TABLE.dsv` (plain production volumes) is intentionally excluded
from this notebook — this notebook is scoped to the disposition table only.

### 2.1 Configuration

Edit the four paths/settings below to match your environment. Nothing else in Part 1 needs to
change for a normal run.

- `OUTER_ZIP` — path to the `texas_pdq.zip` file you downloaded in Section 0.
- `INNER_ZIP` — name of the zip file nested inside the outer zip (`PDQ_DSV.zip`, per RRC's naming).
- `OUT_DIR` — folder where cleaned output files are written. This is the same folder later
  notebooks in the pipeline read from.
- `FORMAT` — `"parquet"` (recommended — smaller, faster, preserves dtypes) or `"csv"`.
- `CHUNKSIZE` — number of raw rows read per chunk. The disposition table has many more columns per
  row than the well table, so this is set lower (150,000) than Part 2's chunk size to keep peak
  memory manageable on a machine with ~16 GB RAM.
- `OIL_GAS_FILTER` — `"O"` restricts to oil leases only (matches the rest of the pipeline), `"G"`
  would restrict to gas leases, `None` disables the filter entirely.

In [2]:
# ── Part 1 config ─────────────────────────────────────────────────────────────

OUTER_ZIP    = "../../../../data/raw/texas/texas_pdq.zip"     # ← path to your outer zip
INNER_ZIP    = "PDQ_DSV.zip"                                  # name of zip inside outer zip
OUT_DIR      = "../../../../data/processed/texas/"      # where to save outputs
FORMAT       = "parquet"                                      # "parquet" or "csv"
CHUNKSIZE    = 150_000                                        # reduced for 16GB RAM
OIL_GAS_FILTER = "O"                                           # "O" = oil leases only
                                                                # "G" = gas leases only
                                                                # None = load everything

log.info("✓ Part 1 config set.")


09:37:37 [INFO] ✓ Part 1 config set.


### 2.2 Constants

These define exactly which raw columns we keep from `OG_LEASE_CYCLE_DISP_DATA_TABLE.dsv`, and how
we rename the cryptic disposition-code columns into readable names.

- `DELIMITER` / `ENCODING` — the `.dsv` files are `}`-delimited and `latin-1` encoded (not UTF-8 —
  this matters, since RRC data occasionally contains non-UTF-8 bytes).
- `DISP_FILE` — the exact filename inside `PDQ_DSV.zip` we're reading.
- `OIL_DISP_LABELS` — maps raw oil disposition codes (e.g. `LEASE_OIL_DISPCD00_VOL`) to readable
  names (e.g. `oil_pipeline_bbl`). Each RRC disposition code corresponds to a specific disposal
  method (pipeline, truck, tank car, lost/stolen, etc.) — see the RRC's Form PR reporting
  instructions for the authoritative code definitions.
- `CSGD_DISP_LABELS` — same idea, for casinghead gas disposition codes (flared, gas lift, sent to
  processing plant, etc.).
- `DISP_KEYS` — the identifying/key columns kept alongside the disposition volumes (lease, district,
  operator, field, month).
- `DISP_KEEP` — the full set of raw columns to keep = keys + all disposition code columns, before
  renaming.

In [3]:
# ── Part 1 constants ──────────────────────────────────────────────────────────

DELIMITER  = "}"
ENCODING   = "latin-1"
DISP_FILE  = "OG_LEASE_CYCLE_DISP_DATA_TABLE.dsv"

OIL_DISP_LABELS = {
    "LEASE_OIL_DISPCD00_VOL": "oil_pipeline_bbl",
    "LEASE_OIL_DISPCD01_VOL": "oil_truck_bbl",
    "LEASE_OIL_DISPCD02_VOL": "oil_tankcar_bbl",
    "LEASE_OIL_DISPCD03_VOL": "oil_tank_cleaning_bbl",
    "LEASE_OIL_DISPCD04_VOL": "oil_circulating_bbl",
    "LEASE_OIL_DISPCD05_VOL": "oil_lost_stolen_bbl",
    "LEASE_OIL_DISPCD06_VOL": "oil_bsw_repressure_bbl",
    "LEASE_OIL_DISPCD07_VOL": "oil_legacy_bbl",
    "LEASE_OIL_DISPCD08_VOL": "oil_skimmed_bbl",
    "LEASE_OIL_DISPCD09_VOL": "oil_scrubber_bbl",
    "LEASE_OIL_DISPCD99_VOL": "oil_no_disp_code_bbl",
}

CSGD_DISP_LABELS = {
    "LEASE_CSGD_DISPCDE01_VOL": "csgd_field_ops_fuel_mcf",
    "LEASE_CSGD_DISPCDE02_VOL": "csgd_transmission_mcf",
    "LEASE_CSGD_DISPCDE03_VOL": "csgd_processing_plant_mcf",
    "LEASE_CSGD_DISPCDE04_VOL": "csgd_vented_flared_mcf",
    "LEASE_CSGD_DISPCDE05_VOL": "csgd_gas_lift_mcf",
    "LEASE_CSGD_DISPCDE06_VOL": "csgd_repressure_mcf",
    "LEASE_CSGD_DISPCDE07_VOL": "csgd_carbon_black_mcf",
    "LEASE_CSGD_DISPCDE08_VOL": "csgd_underground_storage_mcf",
    "LEASE_CSGD_DISPCDE99_VOL": "csgd_no_disp_code_mcf",
}

ALL_DISP_LABELS = {
    **OIL_DISP_LABELS, **CSGD_DISP_LABELS,
}

DISP_KEYS = [
    "OIL_GAS_CODE",
    "DISTRICT_NO",
    "LEASE_NO",
    "CYCLE_YEAR_MONTH",
    "FIELD_NO",
    "OPERATOR_NO",
    "OPERATOR_NAME",
]
DISP_KEEP = DISP_KEYS + list(ALL_DISP_LABELS.keys())

log.info("✓ Part 1 constants defined. %d disposition columns tracked.", len(ALL_DISP_LABELS))


09:37:37 [INFO] ✓ Part 1 constants defined. 20 disposition columns tracked.


### 2.3 Cleaning Functions

Small, composable functions applied to each chunk as it's read (see Section 2.4 for the reader that
calls these):

- `_strip_cols` — strips whitespace from column names and from every string-valued cell (RRC data is
  fixed-width-ish and often has trailing spaces).
- `_apply_filter` — restricts rows to `OIL_GAS_FILTER` (oil leases only, by default).
- `_cast_disp` — converts disposition volume columns to numeric, casts `CYCLE_YEAR_MONTH` to a
  nullable integer, and downcasts a couple of low-cardinality columns to `category` dtype to save
  memory.
- `_add_derived_disp_cols` — computes roll-up totals that aren't in the raw data but are useful
  downstream: `oil_sold_total_bbl` (pipeline + truck + tank car) and, where the relevant gas-well
  columns are present, `total_vented_flared_mcf` / `total_gas_to_processing_mcf`. (Since this
  notebook filters to oil leases only, the casinghead-gas-only columns dominate these totals.)
- `clean_disp_chunk` — orchestrates the above for a single chunk: strip → filter → keep only
  relevant columns → cast → add derived columns → rename disposition codes to readable names. This
  is the function actually passed to the chunked reader in Section 2.4.

In [4]:
# ── Part 1 cleaning functions ─────────────────────────────────────────────────

def _strip_cols(df):
    df.columns = df.columns.str.strip()
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())
    return df

def _apply_filter(df):
    if OIL_GAS_FILTER and "OIL_GAS_CODE" in df.columns:
        df = df[df["OIL_GAS_CODE"].str.strip() == OIL_GAS_FILTER]
    return df

def _cast_disp(df):
    for col in [c for c in df.columns if "_DISPCD" in c or "_DISPCDE" in c]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["CYCLE_YEAR_MONTH"] = pd.to_numeric(
        df["CYCLE_YEAR_MONTH"], errors="coerce").astype("Int32")
    for col in ("OIL_GAS_CODE", "DISTRICT_NO"):
        if col in df.columns:
            df[col] = df[col].astype("category")
    return df

def _add_derived_disp_cols(df):
    present = set(df.columns)
    sold = ["LEASE_OIL_DISPCD00_VOL", "LEASE_OIL_DISPCD01_VOL", "LEASE_OIL_DISPCD02_VOL"]
    if all(c in present for c in sold):
        df["oil_sold_total_bbl"] = df[sold].sum(axis=1, min_count=1)
    g_flare, cg_flare = "LEASE_GAS_DISPCD04_VOL", "LEASE_CSGD_DISPCDE04_VOL"
    if g_flare in present and cg_flare in present:
        df["total_vented_flared_mcf"] = df[[g_flare, cg_flare]].sum(axis=1, min_count=1)
    elif g_flare in present:
        df["total_vented_flared_mcf"] = df[g_flare]
    elif cg_flare in present:
        df["total_vented_flared_mcf"] = df[cg_flare]
    g_proc, cg_proc = "LEASE_GAS_DISPCD03_VOL", "LEASE_CSGD_DISPCDE03_VOL"
    if g_proc in present and cg_proc in present:
        df["total_gas_to_processing_mcf"] = df[[g_proc, cg_proc]].sum(axis=1, min_count=1)
    return df

def clean_disp_chunk(chunk):
    chunk = _strip_cols(chunk)
    chunk = _apply_filter(chunk)
    if chunk.empty:
        return chunk
    chunk = chunk[[c for c in DISP_KEEP if c in chunk.columns]]
    chunk = _cast_disp(chunk)
    chunk = _add_derived_disp_cols(chunk)
    chunk = chunk.rename(columns={
        k: v for k, v in ALL_DISP_LABELS.items() if k in chunk.columns})
    return chunk

log.info("✓ Part 1 cleaning functions defined.")


09:37:37 [INFO] ✓ Part 1 cleaning functions defined.


### 2.4 Reader Functions (Shared with Part 2)

These two functions are generic — they know nothing about disposition data or well data
specifically, which is why they're reused unchanged in Part 2 below instead of being redefined:

- `open_inner_zip(outer_path, inner_name)` — opens the outer zip, finds the inner zip by name
  (case-insensitively), and reads it fully into memory as a `BytesIO` buffer, returning a
  `ZipFile` object over it. This avoids ever writing the inner zip to disk.
- `read_chunked(inner_zf, filename, cleaner)` — opens one `.dsv` file inside the (in-memory) inner
  zip, and streams it through `pandas.read_csv` in chunks of `CHUNKSIZE` rows. Each chunk is passed
  through the supplied `cleaner` function (e.g. `clean_disp_chunk` or, later, `clean_well_chunk`),
  and only the cleaned/filtered result is kept in memory — the raw chunk is discarded and
  garbage-collected immediately. Progress is logged after every chunk so you can monitor how far
  through a multi-gigabyte file you are.

In [5]:
# ── Shared reader functions (used by both Part 1 and Part 2) ────────────────

def open_inner_zip(outer_path, inner_name):
    with zipfile.ZipFile(outer_path, "r") as outer:
        outer_contents = outer.namelist()
        log.info("Files in outer zip: %s", outer_contents)
        match = next((f for f in outer_contents
                      if f.upper() == inner_name.upper()), None)
        if match is None:
            raise FileNotFoundError(
                f"'{inner_name}' not found in outer zip.\nAvailable: {outer_contents}")
        log.info("Opening inner zip: %s", match)
        inner_bytes = BytesIO(outer.read(match))
    return zipfile.ZipFile(inner_bytes, "r")

def read_chunked(inner_zf, filename, cleaner):
    try:
        inner_zf.getinfo(filename)
    except KeyError:
        raise FileNotFoundError(
            f"'{filename}' not found.\nAvailable: {inner_zf.namelist()}")
    log.info("Reading %s (chunk size = %s) ...", filename, f"{CHUNKSIZE:,}")
    chunks, total_in, total_out = [], 0, 0
    with inner_zf.open(filename) as raw:
        reader = pd.read_csv(
            TextIOWrapper(raw, encoding=ENCODING),
            sep=DELIMITER,
            dtype=str,
            chunksize=CHUNKSIZE,
            low_memory=False,
            on_bad_lines="warn",
        )
        for i, chunk in enumerate(reader, 1):
            total_in += len(chunk)
            cleaned = cleaner(chunk)
            if not cleaned.empty:
                chunks.append(cleaned)
                total_out += len(cleaned)
            del chunk, cleaned
            gc.collect()
            log.info("  chunk %3d — kept %s / %s rows",
                     i, f"{total_out:,}", f"{total_in:,}")
    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()
    log.info("✓ Done: %s rows x %s columns", f"{len(df):,}", len(df.columns))
    return df

log.info("✓ Shared reader functions defined.")


09:37:37 [INFO] ✓ Shared reader functions defined.


### 2.5 Load & Inspect `OG_LEASE_CYCLE_DISP_DATA_TABLE`

Opens the inner zip, streams `OG_LEASE_CYCLE_DISP_DATA_TABLE.dsv` through `read_chunked` using
`clean_disp_chunk` as the per-chunk cleaner, then closes the zip handle and frees memory. This is
the slowest cell in Part 1 — expect it to take a while given the file's size — so the shape, memory
footprint, and a duplicate-column sanity check are printed immediately after for a quick health
check.

In [6]:
# ── Load OG_LEASE_CYCLE_DISP ─────────────────────────────────────────────────

log.info("\n── Loading OG_LEASE_CYCLE_DISP ──")
inner_zf = open_inner_zip(OUTER_ZIP, INNER_ZIP)
df_disp = read_chunked(inner_zf, DISP_FILE, clean_disp_chunk)
inner_zf.close()
gc.collect()

print("\nShape  :", df_disp.shape)
print("Memory :", f"{df_disp.memory_usage(deep=True).sum() / 1_048_576:.1f} MB")
print("\nColumns:", df_disp.columns.tolist())

dupes = df_disp.columns[df_disp.columns.duplicated()].tolist()
if dupes:
    log.error("Duplicate columns in df_disp: %s", dupes)
else:
    log.info("✓ No duplicate columns.")

df_disp.head(3)


09:37:38 [INFO] 
── Loading OG_LEASE_CYCLE_DISP ──
09:37:38 [INFO] Files in outer zip: ['PDQ_DSV.zip']
09:37:38 [INFO] Opening inner zip: PDQ_DSV.zip
09:37:43 [INFO] Reading OG_LEASE_CYCLE_DISP_DATA_TABLE.dsv (chunk size = 150,000) ...
09:37:44 [INFO]   chunk   1 — kept 47,657 / 150,000 rows
09:37:44 [INFO]   chunk   2 — kept 108,852 / 300,000 rows
09:37:45 [INFO]   chunk   3 — kept 148,948 / 450,000 rows
09:37:46 [INFO]   chunk   4 — kept 195,204 / 600,000 rows
09:37:47 [INFO]   chunk   5 — kept 233,180 / 750,000 rows
09:37:47 [INFO]   chunk   6 — kept 283,864 / 900,000 rows
09:37:48 [INFO]   chunk   7 — kept 346,512 / 1,050,000 rows
09:37:49 [INFO]   chunk   8 — kept 392,278 / 1,200,000 rows
09:37:49 [INFO]   chunk   9 — kept 481,437 / 1,350,000 rows
09:37:50 [INFO]   chunk  10 — kept 525,218 / 1,500,000 rows
09:37:51 [INFO]   chunk  11 — kept 591,342 / 1,650,000 rows
09:37:52 [INFO]   chunk  12 — kept 666,076 / 1,800,000 rows
09:37:52 [INFO]   chunk  13 — kept 710,671 / 1,950,000 ro

,OIL_GAS_CODE,DISTRICT_NO,LEASE_NO,CYCLE_YEAR_MONTH,FIELD_NO,OPERATOR_NO,OPERATOR_NAME,oil_pipeline_bbl,oil_truck_bbl,oil_tankcar_bbl,...,csgd_transmission_mcf,csgd_processing_plant_mcf,csgd_vented_flared_mcf,csgd_gas_lift_mcf,csgd_repressure_mcf,csgd_carbon_black_mcf,csgd_underground_storage_mcf,csgd_no_disp_code_mcf,oil_sold_total_bbl,total_vented_flared_mcf
0,O,08,09073,202603,89812001,101688,"FOURCOOKS OIL & GAS, LLC",0,165,0,...,0,0,0,0,0,0,0,0,165,0
1,O,10,27510,202601,19541001,623254,ONE NICKEL OPERATING LLC,0,146,0,...,0,0,0,0,0,0,0,0,146,0
2,O,10,27510,202602,19541001,623254,ONE NICKEL OPERATING LLC,0,154,0,...,0,0,0,0,0,0,0,0,154,0


### 2.6 Derive a Proper `date` Column

`CYCLE_YEAR_MONTH` is a raw integer like `202401` (YYYYMM). We parse it into an actual pandas
`datetime` column called `date`, which is what all downstream notebooks join and filter on.

In [7]:
#Make a date_time column
print(df_disp.columns)
df_disp['date'] = pd.to_datetime(df_disp['CYCLE_YEAR_MONTH'],format = '%Y%m')
df_disp.date.head(3)


Index(['OIL_GAS_CODE', 'DISTRICT_NO', 'LEASE_NO', 'CYCLE_YEAR_MONTH',
       'FIELD_NO', 'OPERATOR_NO', 'OPERATOR_NAME', 'oil_pipeline_bbl',
       'oil_truck_bbl', 'oil_tankcar_bbl', 'oil_tank_cleaning_bbl',
       'oil_circulating_bbl', 'oil_lost_stolen_bbl', 'oil_bsw_repressure_bbl',
       'oil_legacy_bbl', 'oil_skimmed_bbl', 'oil_scrubber_bbl',
       'oil_no_disp_code_bbl', 'csgd_field_ops_fuel_mcf',
       'csgd_transmission_mcf', 'csgd_processing_plant_mcf',
       'csgd_vented_flared_mcf', 'csgd_gas_lift_mcf', 'csgd_repressure_mcf',
       'csgd_carbon_black_mcf', 'csgd_underground_storage_mcf',
       'csgd_no_disp_code_mcf', 'oil_sold_total_bbl',
       'total_vented_flared_mcf'],
      dtype='str')


0   2026-03-01
1   2026-01-01
2   2026-02-01
Name: date, dtype: datetime64[us]

### 2.7 Drop the Now-Redundant `CYCLE_YEAR_MONTH` Column

Now that we have a proper `date` column, the raw `CYCLE_YEAR_MONTH` integer is redundant and is
dropped to keep the output tidy.

In [8]:
df_disp = df_disp.drop(columns = ['CYCLE_YEAR_MONTH'])


### 2.8 Drop Rows With No Recorded Oil Sales

Rows where the derived `oil_sold_total_bbl` is null (no oil was reported sold that month for that
lease) are dropped, since they carry no useful disposition signal for this dataset. A fresh copy is
made afterward to avoid holding a view over the original frame.

In [9]:
# Cleaning further
df_disp_clean = df_disp.dropna(subset = 'oil_sold_total_bbl').reset_index(drop=True).copy()
df_disp_clean.head(3)


,OIL_GAS_CODE,DISTRICT_NO,LEASE_NO,FIELD_NO,OPERATOR_NO,OPERATOR_NAME,oil_pipeline_bbl,oil_truck_bbl,oil_tankcar_bbl,oil_tank_cleaning_bbl,...,csgd_processing_plant_mcf,csgd_vented_flared_mcf,csgd_gas_lift_mcf,csgd_repressure_mcf,csgd_carbon_black_mcf,csgd_underground_storage_mcf,csgd_no_disp_code_mcf,oil_sold_total_bbl,total_vented_flared_mcf,date
0,O,08,09073,89812001,101688,"FOURCOOKS OIL & GAS, LLC",0,165,0,0,...,0,0,0,0,0,0,0,165,0,2026-03-01
1,O,10,27510,19541001,623254,ONE NICKEL OPERATING LLC,0,146,0,0,...,0,0,0,0,0,0,0,146,0,2026-01-01
2,O,10,27510,19541001,623254,ONE NICKEL OPERATING LLC,0,154,0,0,...,0,0,0,0,0,0,0,154,0,2026-02-01


### 2.9 Lowercase Column Names

Column names are lowercased for consistency with the rest of the pipeline (all downstream notebooks
expect lowercase snake_case columns, e.g. `oil_gas_code`, `lease_no`, `oil_sold_total_bbl`).

In [10]:
df_disp_clean.columns = df_disp_clean.columns.str.lower()


### 2.10 Save Part 1 Output

Writes the cleaned disposition table to `OUT_DIR` as `texas_prod_disp.<FORMAT>`. The `save_df`
helper defined here is reused again in Part 2 to save the well/lease/API table. After saving, the
large intermediate frames are explicitly deleted and garbage-collected to free memory before Part 2
runs.

**Resulting columns in `texas_prod_disp.parquet`:**

| Column | Description |
|---|---|
| `oil_gas_code` | Always `"O"` (oil leases only) |
| `district_no` | RRC district number |
| `lease_no` | RRC lease number |
| `field_no` | RRC field number |
| `operator_no` | RRC operator ID |
| `operator_name` | Operator name |
| `date` | Production month (datetime) |
| `oil_pipeline_bbl`, `oil_truck_bbl`, `oil_tankcar_bbl`, `oil_tank_cleaning_bbl`, `oil_circulating_bbl`, `oil_lost_stolen_bbl`, `oil_bsw_repressure_bbl`, `oil_legacy_bbl`, `oil_skimmed_bbl`, `oil_scrubber_bbl`, `oil_no_disp_code_bbl` | Oil disposition breakdown (BBL) |
| `oil_sold_total_bbl` | Derived: pipeline + truck + tank car (BBL) |
| `csgd_field_ops_fuel_mcf`, `csgd_transmission_mcf`, `csgd_processing_plant_mcf`, `csgd_vented_flared_mcf`, `csgd_gas_lift_mcf`, `csgd_repressure_mcf`, `csgd_carbon_black_mcf`, `csgd_underground_storage_mcf`, `csgd_no_disp_code_mcf` | Casinghead gas disposition breakdown (MCF) |
| `total_vented_flared_mcf` | Derived: total gas vented/flared (MCF) — equals `csgd_vented_flared_mcf` for this oil-only pipeline |

> Note: a `total_gas_to_processing_mcf` roll-up is computed by the derived-columns logic for leases
> that report *both* oil-well and gas-well processing-plant codes. Since this pipeline filters to oil
> leases only, that second code is never present, so this column will not appear in the saved output
> — it's mentioned here only so the logic in Section 2.3 doesn't look incomplete.

In [11]:
# Saving this file
out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)

def save_df(df, name):
    path = out / f"{name}.{FORMAT}"
    if FORMAT == "parquet":
        df.to_parquet(path, index=False)
    else:
        df.to_csv(path, index=False)
    log.info("Saved %s  (%.1f MB)", path.name, path.stat().st_size / 1_048_576)

log.info("Saving df_disp ...")
save_df(df_disp_clean, "texas_prod_disp")
del df_disp
del df_disp_clean
gc.collect()
log.info("✓ df_disp_clean saved and freed from memory.")


09:41:30 [INFO] Saving df_disp ...
09:41:35 [INFO] Saved texas_prod_disp.parquet  (157.5 MB)
09:41:35 [INFO] ✓ df_disp_clean saved and freed from memory.


## 3. Part 2 — Unique Well ↔ Lease ↔ API Identification (`OG_WELL_COMPLETION_DATA_TABLE`)

**Goal:** build the mapping between individual wells and the lease they belong to, and construct
each well's 8-digit RRC API number.

Texas RRC's public production data (Part 1 above) is reported at the **lease** level, not the well
level — a single lease can contain multiple wells, and RRC does not publish how much each individual
well within a lease produced. To eventually locate wells spatially (in a later pipeline notebook) and
approximate well-level production, we first need to know **which wells belong to which lease**, and
each well's **API number** — the RRC's unique well identifier, which is also the join key used by
RRC's well-location shapefiles.

The API number is built from two raw fields: a 3-digit county code and a 5-digit unique well number
within that county, zero-padded and concatenated into an 8-digit `API_NO` (commonly called "API8").

**Output of Part 2:** `well_api_lease.parquet` (or `.csv`), ~587k rows, one per oil well.

This section reuses the generic `open_inner_zip` and `read_chunked` functions defined in Section 2.4
— they are not redefined here.

### 3.1 Configuration

Same idea as Part 1's config (Section 2.1), but with its own `CHUNKSIZE`. The well completion table
has far fewer columns per row than the disposition table, so a larger chunk size (75,000) is safe
here without blowing up memory.

- `OUTER_ZIP` / `INNER_ZIP` — same source zip as Part 1 (same download from Section 0).
- `OUT_DIR` — same output folder as Part 1, so both outputs land side by side.
- `FORMAT` — should match Part 1's format for consistency, though this isn't strictly required.
- `CHUNKSIZE` — 75,000 rows per chunk (vs. 150,000 in Part 1) since this table is narrower.
- `OIL_GAS_FILTER` — should match Part 1's filter (`"O"`) so the well table and disposition table
  cover the same population of leases.

In [16]:
# ── Part 2 config ─────────────────────────────────────────────────────────────

OUTER_ZIP      = "../../../../data/raw/texas/texas_pdq.zip"          # your outer zip
INNER_ZIP      = "PDQ_DSV.zip"                                       # zip inside outer
OUT_DIR        = "../../../../data/processed/texas"           # same folder as Part 1 outputs
FORMAT         = "parquet"
CHUNKSIZE      = 75_000
OIL_GAS_FILTER = "O"                                                  # must match Part 1's filter

# Constants
DELIMITER  = "}"
ENCODING   = "latin-1"
WELL_FILE  = "OG_WELL_COMPLETION_DATA_TABLE.dsv"

# Only the columns we need — keeps memory very low
WELL_KEEP = [
    "OIL_GAS_CODE",
    "DISTRICT_NO",
    "LEASE_NO",       # joins to og_lease_cycle / texas_prod_disp via lease_no + district_no
    "WELL_NO",
    "API_COUNTY_CODE",
    "API_UNIQUE_NO",  # combined with API_COUNTY_CODE → API_NO → coordinates
    "COUNTY_NAME",
    "WELLBORE_LOCATION_CODE",
]

log.info("✓ Part 2 config set.")


09:43:36 [INFO] ✓ Part 2 config set.


### 3.2 Cleaning Function

`clean_well_chunk` mirrors the structure of Part 1's `clean_disp_chunk`, but is specific to the well
completion table:

1. Strip whitespace from column names and string values (`_strip_cols`, reused from Part 1).
2. Filter to oil wells only (`_apply_filter`, reused from Part 1).
3. Keep only the columns listed in `WELL_KEEP`.
4. Construct the 8-digit `API_NO` by zero-padding `API_COUNTY_CODE` to 3 digits and `API_UNIQUE_NO`
   to 5 digits, then concatenating them. This exact 8-digit format (**API8**) is what matches the RRC
   well-location shapefiles used in the next pipeline notebook.
5. Downcast a few low-cardinality columns to `category` dtype to save memory.

Note that `_strip_cols` and `_apply_filter` are **not redefined** here — they were already defined in
Section 2.3 and are reused as-is, since the logic is identical for both tables.

In [17]:
# ── Part 2 cleaning function ──────────────────────────────────────────────────
# Reuses _strip_cols and _apply_filter from Part 1 (Section 2.3), and open_inner_zip /
# read_chunked from Section 2.4 — no need to redefine them here.

def clean_well_chunk(chunk):
    chunk = _strip_cols(chunk)
    chunk = _apply_filter(chunk)
    if chunk.empty:
        return chunk
    chunk = chunk[[c for c in WELL_KEEP if c in chunk.columns]]
    # Build full 8-digit API number
    if "API_COUNTY_CODE" in chunk.columns and "API_UNIQUE_NO" in chunk.columns:
        chunk["API_NO"] = (
            chunk["API_COUNTY_CODE"].str.strip().str.zfill(3) +
            chunk["API_UNIQUE_NO"].str.strip().str.zfill(5)
        )
    for col in ("OIL_GAS_CODE", "DISTRICT_NO", "WELLBORE_LOCATION_CODE"):
        if col in chunk.columns:
            chunk[col] = chunk[col].astype("category")
    return chunk

log.info("✓ Part 2 cleaning function defined.")


09:43:38 [INFO] ✓ Part 2 cleaning function defined.


### 3.3 Load & Inspect `OG_WELL_COMPLETION_DATA_TABLE`

Same load pattern as Part 1 (Section 2.5): open the inner zip, stream the well completion table
through `read_chunked` with `clean_well_chunk`, close the zip, and inspect the result. A quick sanity
check counts how many wells ended up with a valid (non-placeholder) `API_NO`.

In [18]:
# ── Load OG_WELL_COMPLETION ───────────────────────────────────────────────────

log.info("\n── Loading OG_WELL_COMPLETION ──")
inner_zf = open_inner_zip(OUTER_ZIP, INNER_ZIP)
df_well  = read_chunked(inner_zf, WELL_FILE, clean_well_chunk)
inner_zf.close()
gc.collect()

print("\nShape  :", df_well.shape)
print("Memory :", f"{df_well.memory_usage(deep=True).sum() / 1_048_576:.1f} MB")
print("\nColumns:", df_well.columns.tolist())
print("\nSample:")
print(df_well[["DISTRICT_NO", "LEASE_NO", "WELL_NO",
               "API_NO", "COUNTY_NAME"]].head(10))

# Quick check — how many have a valid API number
has_api = df_well["API_NO"].notna() & (df_well["API_NO"] != "00000000")
print(f"\nWells with valid API_NO: {has_api.sum():,} / {len(df_well):,}")


09:43:38 [INFO] 
── Loading OG_WELL_COMPLETION ──
09:43:38 [INFO] Files in outer zip: ['PDQ_DSV.zip']
09:43:38 [INFO] Opening inner zip: PDQ_DSV.zip
09:43:43 [INFO] Reading OG_WELL_COMPLETION_DATA_TABLE.dsv (chunk size = 75,000) ...
09:43:43 [INFO]   chunk   1 — kept 0 / 75,000 rows
09:43:43 [INFO]   chunk   2 — kept 0 / 150,000 rows
09:43:43 [INFO]   chunk   3 — kept 495 / 225,000 rows
09:43:43 [INFO]   chunk   4 — kept 68,669 / 300,000 rows
09:43:43 [INFO]   chunk   5 — kept 143,669 / 375,000 rows
09:43:44 [INFO]   chunk   6 — kept 218,669 / 450,000 rows
09:43:44 [INFO]   chunk   7 — kept 293,669 / 525,000 rows
09:43:44 [INFO]   chunk   8 — kept 368,669 / 600,000 rows
09:43:44 [INFO]   chunk   9 — kept 443,669 / 675,000 rows
09:43:44 [INFO]   chunk  10 — kept 518,669 / 750,000 rows
09:43:44 [INFO]   chunk  11 — kept 587,614 / 818,945 rows
09:43:44 [INFO] ✓ Done: 587,614 rows x 9 columns

Shape  : (587614, 9)
Memory : 55.1 MB

Columns: ['OIL_GAS_CODE', 'DISTRICT_NO', 'LEASE_NO', 'WELL

### 3.4 Save Part 2 Output

Writes the well/lease/API mapping to `OUT_DIR` as `well_api_lease.<FORMAT>`, using the same naming
convention as Part 1's output. Lists everything currently in the output folder afterward as a final
sanity check that both this notebook's outputs are present.

**Resulting columns in `well_api_lease.parquet`:**

| Column | Type | Description |
|---|---|---|
| `oil_gas_code` | category | Always `"O"` |
| `district_no` | category | RRC district number |
| `lease_no` | str | RRC lease number |
| `well_no` | str | Well number, unique within a lease |
| `api_county_code` | str | 3-digit RRC/API county code |
| `api_unique_no` | str | 5-digit unique well number within the county |
| `county_name` | str | County name |
| `wellbore_location_code` | category | `L` = Land, `O` = Offshore, `I` = Inland Waterway, `B` = Bay/Estuary |
| `api_no` | str | Constructed 8-digit API number (API8) |

In [19]:
# ── Part 2 save ───────────────────────────────────────────────────────────────

out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)

path = out / f"well_api_lease.{FORMAT}"
if FORMAT == "parquet":
    df_well.to_parquet(path, index=False)
else:
    df_well.to_csv(path, index=False)

size_mb = path.stat().st_size / 1_048_576
log.info("✓ Saved %s  (%.1f MB)", path.name, size_mb)

print("\nAll files in output folder:")
for f in sorted(out.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1_048_576:.1f} MB)")


09:43:44 [INFO] ✓ Saved well_api_lease.parquet  (6.4 MB)

All files in output folder:
  cleaned_data  (0.0 MB)
  texas_prod_disp.parquet  (157.5 MB)
  well_api_lease.parquet  (6.4 MB)


## 4. Summary 

This notebook produced two files in `OUT_DIR`:

| File | Rows (approx.) | Description |
|---|---|---|
| `texas_prod_disp.parquet` | Millions (monthly, per lease, 1993–present) | Oil/casinghead gas disposition breakdown per lease per month |
| `well_api_lease.parquet` | ~587k | One row per oil well, mapped to its lease, with API8 identifier |